In [1]:
# ===== M1：数据准备与动量因子构造 =====
%pip install akshare
import akshare as ak
import pandas as pd
import numpy as np

# --- 方案A：真实数据（akshare 拉沪深300成分股月度行情）---
try:
    hs300 = ak.index_stock_cons_csindex(symbol="000300")
    codes = hs300["成分券代码"].tolist()
    all_data = []
    for code in codes[:50]:  # 先取50只演示，全市场去掉切片
        df = ak.stock_zh_a_hist(symbol=code, period="monthly",
                                start_date="20150101", end_date="20241231", adjust="qfq")
        df["code"] = code
        all_data.append(df)
    panel = pd.concat(all_data)
    panel["date"] = pd.to_datetime(panel["日期"])
    panel = panel.rename(columns={"收盘": "close"})
    USE_REAL = True
except Exception as e:
    print("akshare 拉取失败，使用模拟数据：", e)
    USE_REAL = False

# --- 方案B：模拟数据兜底（确保代码可运行）---
if not USE_REAL:
    np.random.seed(42)
    n_stocks, n_months = 50, 120
    dates = pd.date_range("2015-01-31", periods=n_months, freq="ME")
    codes = [f"{i:06d}" for i in range(n_stocks)]
    rows = []
    for code in codes:
        beta = np.random.uniform(0.3, 1.2)
        prices = 100 * np.cumprod(1 + np.random.normal(0.01*beta, 0.08, n_months))
        for d, p in zip(dates, prices):
            rows.append({"code": code, "date": d, "close": p})
    panel = pd.DataFrame(rows)

# --- 月度对数收益率 ---
panel = panel.sort_values(["code", "date"])
panel["ret"] = panel.groupby("code")["close"].transform(
    lambda x: np.log(x / x.shift(1))
)

# --- 动量因子：过去12-1个月累计收益 ---
panel["mom_12_1"] = panel.groupby("code")["ret"].transform(
    lambda x: x.shift(1).rolling(11).sum()
)

# --- 剔除缺失 ---
panel = panel.dropna(subset=["ret", "mom_12_1"])
print(f"样本：{panel['code'].nunique()} 只股票，{panel['date'].nunique()} 个月，{len(panel)} 条观测")
panel.to_csv("momentum_panel.csv", index=False)

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.
akshare 拉取失败，使用模拟数据： ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
样本：50 只股票，108 个月，5400 条观测


In [3]:
# ===== M2：基准分析 =====
import statsmodels.api as sm

panel = pd.read_csv("momentum_panel.csv", parse_dates=["date"])

# --- 分组法：每月按动量分5组 ---
def port_sort(p, factor, n=5):
    p = p.copy()
    p["grp"] = p.groupby("date")[factor].transform(
        lambda x: pd.qcut(x, n, labels=False, duplicates="drop")
    )
    means = p.groupby(["date", "grp"])["ret"].mean().unstack()
    return means

means = port_sort(panel, "mom_12_1", 5)
avg_means = means.mean()
hl_series = means.iloc[:, -1] - means.iloc[:, 0]
t_hl = hl_series.mean() / (hl_series.std() / np.sqrt(len(hl_series)))

print("各组平均月收益：")
print(avg_means.round(4))
print(f"H-L 月收益: {hl_series.mean():.4f}, t值: {t_hl:.3f}")

# --- Fama-MacBeth 两步法 ---
coefs = []
for t in panel["date"].unique():
    sub = panel[panel["date"] == t]
    if len(sub) <10:
        continue
    X = sm.add_constant(sub["mom_12_1"])
    res = sm.OLS(sub["ret"], X).fit()
    coefs.append(res.params["mom_12_1"])

fm = pd.Series(coefs)
t_fm = fm.mean() / (fm.std() / np.sqrt(len(fm)))
print(f"\nFM 风险溢价: {fm.mean():.4f}, t值: {t_fm:.3f}")

各组平均月收益：
grp
0    0.0022
1    0.0057
2    0.0059
3    0.0011
4    0.0026
dtype: float64
H-L 月收益: 0.0004, t值: 0.112

FM 风险溢价: -0.0023, t值: -0.491


In [6]:
# ===== M3：诊断与稳健性 =====
from statsmodels.stats.sandwich_covariance import cov_hac

# --- 稳健性1：Newey-West 标准误（FM 系数序列相关调整）---
def newey_west_se(x, nlags=6):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    T = len(x)
    gamma0 = (x @ x) / T
    var = gamma0
    for j in range(1, nlags + 1):
        gamma_j = (x[j:] @ x[:-j]) / T
        w = 1 - j / (nlags + 1)          # Bartlett 权重
        var += 2 * w * gamma_j
    return np.sqrt(var / T)

nw_se = newey_west_se(fm.values, nlags=6)
t_fm_nw = fm.mean() / nw_se
print(f"FM 风险溢价(Newey-West): {fm.mean():.4f}, t值: {t_fm_nw:.3f}")

# --- 稳健性2：换分组数（10组）---
means10 = port_sort(panel, "mom_12_1", 10)
hl10 = means10.iloc[:, -1] - means10.iloc[:, 0]
t_hl10 = hl10.mean() / (hl10.std() / np.sqrt(len(hl10)))
print(f"10分组 H-L: {hl10.mean():.4f}, t值: {t_hl10:.3f}")

# --- 稳健性3：换样本期（2018年后）---
panel_recent = panel[panel["date"] >= "2018-01-01"]
means_r = port_sort(panel_recent, "mom_12_1", 5)
hl_r = means_r.iloc[:, -1] - means_r.iloc[:, 0]
t_hl_r = hl_r.mean() / (hl_r.std() / np.sqrt(len(hl_r)))
print(f"2018后 H-L: {hl_r.mean():.4f}, t值: {t_hl_r:.3f}")

# --- 稳健性4：去极值（因子1%/99%缩尾）---
def winsorize(s, lower=0.01, upper=0.99):
    q1, q99 = s.quantile(lower), s.quantile(upper)
    return s.clip(q1, q99)

panel["mom_w"] = panel.groupby("date")["mom_12_1"].transform(winsorize)
means_w = port_sort(panel, "mom_w", 5)
hl_w = means_w.iloc[:, -1] - means_w.iloc[:, 0]
t_hl_w = hl_w.mean() / (hl_w.std() / np.sqrt(len(hl_w)))
print(f"缩尾后 H-L: {hl_w.mean():.4f}, t值: {t_hl_w:.3f}")

FM 风险溢价(Newey-West): -0.0023, t值: -0.573
10分组 H-L: -0.0007, t值: -0.124
2018后 H-L: -0.0028, t值: -0.689
缩尾后 H-L: 0.0004, t值: 0.112


In [7]:
# ===== M4：深化方法——三因子 alpha =====
# 多空组合收益
hl_returns = means.iloc[:, -1] - means.iloc[:, 0]
hl_returns.name = "hl_ret"

# 三因子数据（真实数据用 Fama-French Data Library 或国内 CSMAR；此处模拟）
np.random.seed(123)
ff = pd.DataFrame({
    "date": hl_returns.index,
    "mkt_rf": np.random.normal(0.008, 0.05, len(hl_returns)),
    "smb": np.random.normal(0.002, 0.03, len(hl_returns)),
    "hml": np.random.normal(0.003, 0.03, len(hl_returns)),
}).set_index("date")

reg_data = pd.concat([hl_returns, ff], axis=1).dropna()
X = sm.add_constant(reg_data[["mkt_rf", "smb", "hml"]])
res = sm.OLS(reg_data["hl_ret"], X).fit(cov_type="HC1")

print("三因子回归结果：")
print(res.summary().tables[1])
print(f"\nalpha: {res.params['const']:.4f}, p值: {res.pvalues['const']:.4f}")
print("alpha 显著 → 动量超额不能被三因子解释")
print("alpha 不显著 → 动量收益可被常见风险因子解释")

三因子回归结果：
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.004      0.150      0.881      -0.007       0.008
mkt_rf        -0.0004      0.064     -0.007      0.995      -0.125       0.124
smb           -0.0784      0.138     -0.569      0.570      -0.348       0.192
hml            0.0258      0.143      0.181      0.857      -0.254       0.305

alpha: 0.0006, p值: 0.8808
alpha 显著 → 动量超额不能被三因子解释
alpha 不显著 → 动量收益可被常见风险因子解释
